## STEP 1. Chapter 04 환경 확인


In [ ]:
from pathlib import Path
import sys
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError(
        "프로젝트 루트 폴더를 찾을 수 없습니다. "
        "Notebook을 llm-data-analysis-course 저장소 안에서 실행해 주세요."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORT_DIR = PROJECT_ROOT / "reports"

print("Python 실행 파일:", sys.executable)
print("현재 작업 폴더:", Path.cwd())
print("프로젝트 루트:", PROJECT_ROOT)
print("데이터 폴더:", DATA_DIR)
print("결과 저장 폴더:", REPORT_DIR)


## STEP 2. 4개 CSV 확인하고 불러오기


In [ ]:
required_files = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
]

for file_name in required_files:
    file_path = DATA_DIR / file_name
    print(file_name, file_path.exists())


In [ ]:
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

print("데이터 불러오기 완료")


## STEP 3. 실제 컬럼명 확인하기


In [ ]:
for name, df in datasets.items():
    print(name, df.shape, df.columns.tolist())


## STEP 4. 컬럼 선택과 행 필터링


In [ ]:
customer_basic = customers[[
    "customer_id",
    "gender",
    "age",
    "city",
]]

customers_over_30 = customers[
    customers["age"] >= 30
]

city_customers = customers[
    customers["city"].isin(["서울", "부산"])
]

print(customers["city"].value_counts())
print(orders["order_status"].value_counts())

display(customer_basic.head())
display(customers_over_30.head())
display(city_customers.head())


## STEP 5. 정렬과 파생 컬럼 만들기


In [ ]:
display(
    products.sort_values(
        "price",
        ascending=False,
    ).head(10)
)

order_items = order_items.copy()

order_items["line_total"] = (
    order_items["quantity"]
    * order_items["unit_price"]
)

display(
    order_items[[
        "order_id",
        "product_id",
        "quantity",
        "unit_price",
        "line_total",
    ]].head()
)


## STEP 6. orders와 병합하고 검증하기


In [ ]:
print(
    "orders.order_id 중복 수:",
    orders["order_id"].duplicated().sum(),
)

order_sales = order_items.merge(
    orders[[
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("병합 전 행 수:", len(order_items))
print("병합 후 행 수:", len(order_sales))
print(order_sales["_merge"].value_counts())


## STEP 7. 날짜 변환 후 completed 주문만 선택하기


In [ ]:
order_sales["order_date"] = pd.to_datetime(
    order_sales["order_date"],
    errors="coerce",
)

print(
    "날짜 변환 실패:",
    order_sales["order_date"].isna().sum(),
)

completed_order_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

completed_order_sales["order_month"] = (
    completed_order_sales["order_date"]
    .dt.to_period("M")
    .astype(str)
)

print(completed_order_sales["order_status"].unique())


## STEP 8. products와 병합하고 다시 검증하기


In [ ]:
completed_sales_items = completed_order_sales.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("병합 전:", len(completed_order_sales))
print("병합 후:", len(completed_sales_items))
print(completed_sales_items["_merge"].value_counts())


## STEP 9. 카테고리별·상품별 요약표 만들기


In [ ]:
category_sales = (
    completed_sales_items
    .groupby("category", as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)

product_sales = (
    completed_sales_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)

display(category_sales)
display(product_sales.head(10))

print(category_sales["total_sales"].sum())
print(completed_sales_items["line_total"].sum())


## STEP 10. 월별·고객별 요약표 만들기


In [ ]:
monthly_summary = (
    completed_order_sales
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
    )
    .sort_values("order_month")
)

customer_sales = (
    completed_order_sales
    .groupby("customer_id", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)

customer_sales = customer_sales.merge(
    customers[["customer_id", "city"]],
    on="customer_id",
    how="left",
    validate="one_to_one",
)

display(monthly_summary)
display(customer_sales.head(10))


## STEP 11. 결과 CSV 저장하고 다시 읽기


In [ ]:
category_sales.to_csv(
    DATA_DIR / "ch04_category_sales.csv",
    index=False
)

product_sales.to_csv(
    DATA_DIR / "ch04_product_sales.csv",
    index=False
)

monthly_summary.to_csv(
    DATA_DIR / "ch04_monthly_sales.csv",
    index=False
)

customer_sales.to_csv(
    DATA_DIR / "ch04_customer_sales.csv",
    index=False
)

print("CSV 저장 완료")

In [ ]:
check_category = pd.read_csv(
    REPORT_DIR / "ch04_category_sales.csv"
)

print(check_category.shape)
print(check_category.columns.tolist())
display(check_category.head())


## STEP 12. LLM 코드 검증하기

다음을 확인합니다.

- 실제 컬럼명을 사용했는가?
- 키 관계가 맞는가?
- completed 필터가 있는가?
- 병합 검증이 있는가?
- 집계 단위가 맞는가?
- 결과 의미를 과장하지 않았는가?


## STEP 13. Chapter 04 Evidence

### 1. Notebook
- notebooks/ch04_pandas_basic.ipynb 실행 완료: 예

### 2. 병합 검증
- orders.order_id 중복: 0
- orders merge 전 행 수: 764
- orders merge 후 행 수: 764
- orders merge 미매칭: 0
- products merge 전 행 수: 실행 결과 입력
- products merge 후 행 수: 실행 결과 입력
- products merge 미매칭: 0

### 3. 계산 범위
- 사용 주문 상태: completed
- line_total 계산식: quantity × unit_price
- 날짜 변환 실패: 실행 결과 입력

### 4. 교차 검증
- 완료 주문 상세 line_total 합계: 실행 결과 입력
- category_sales 합계: 실행 결과 입력
- monthly_summary 합계: 실행 결과 입력
- customer_sales 합계: 실행 결과 입력
- 합계 일치 여부: PASS

### 5. 저장 파일
- ch04_category_sales.csv: 존재
- ch04_product_sales.csv: 존재
- ch04_monthly_sales.csv: 존재
- ch04_customer_sales.csv: 존재

### 6. LLM 검증
- 실제 컬럼명을 사용했는지 확인했다.
- 키 관계가 올바른지 확인했다.
- completed 주문만 분석에 포함했는지 확인했다.
- merge 시 validate와 indicator를 사용했는지 확인했다.
- 병합 전후 행 수와 미매칭 여부를 확인했다.
- 집계 기준과 total_sales 계산 기준이 맞는지 확인했다.

### 7. 남은 질문
- 없음

# 최종 완료 체크리스트

- [ ] Chapter 04 Notebook을 올바른 `.venv` 커널로 실행했다.
- [ ] 4개 CSV를 불러왔다.
- [ ] 실제 컬럼명과 필터 값을 확인했다.
- [ ] `line_total`을 만들었다.
- [ ] orders merge에 `validate`와 `indicator`를 사용했다.
- [ ] 병합 전후 행 수와 미매칭을 확인했다.
- [ ] `completed` 주문만 분석 범위로 사용했다.
- [ ] products merge도 같은 방식으로 검증했다.
- [ ] 카테고리·상품·월·고객 집계를 만들었다.
- [ ] 여러 집계의 `total_sales` 합계를 교차 검증했다.
- [ ] 4개 결과 CSV를 저장하고 다시 확인했다.
- [ ] 고객 결과에서 불필요한 개인정보를 사용하지 않았다.
- [ ] LLM 코드를 실제 데이터 구조와 계산 기준에 대조했다.
- [ ] Evidence를 실제 실행 결과로 작성했다.
